In [1]:
import geobr
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box
import time, sys, os, math
from pyspark.sql import functions as F

In [2]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Importa métodos e/ou funções 
from spark_utils import get_spark_session, write_data_csv, convert_nc_to_spark_dataframe

In [3]:
# Cria uma conexão Spark 
spark = get_spark_session("Municipio_Shape")

In [4]:
def criar_grade_copernicus(min_lon, min_lat, max_lon, max_lat, step=0.25):
    print(f"--> Gerando grade alinhada do Copernicus (Resolução: {step}°)...")
    
    # FORÇA O ALINHAMENTO aos múltiplos exatos de 0.25 (ex: -74.00, -73.75, -73.50...)
    start_lon = np.round(min_lon / step) * step
    end_lon   = np.round(max_lon / step) * step
    start_lat = np.round(min_lat / step) * step
    end_lat   = np.round(max_lat / step) * step

    # Cria os eixos perfeitos
    lons = np.round(np.arange(start_lon, end_lon + step, step), 2)
    lats = np.round(np.arange(start_lat, end_lat + step, step), 2)
    
    grid_cells = []
    half = step / 2.0
    
    for lat in lats:
        for lon in lons:
            cell_poly = box(lon - half, lat - half, lon + half, lat + half)
            grid_cells.append({
                'id_geo': f"GRID_{lat:.2f}_{lon:.2f}",
                'latitude_centro': round(float(lat), 2),   # Ex: -33.75
                'longitude_centro': round(float(lon), 2),  # Ex: -53.50 (e NÃO -53.49)
                'geometry': cell_poly
            })
            
    gdf_grid = gpd.GeoDataFrame(grid_cells, crs="EPSG:4326")
    print(f"    Total de células geradas na grade: {len(gdf_grid):,}")
    return gdf_grid

def calcular_pesos_sobreposicao(gdf_grid, gdf_municipios, crs_metrico="EPSG:5880"):
    """
    Calcula a área de sobreposição entre a grade climática e os municípios em m².
    EPSG:5880 = SIRGAS 2000 / Brasil Polyconic (Ideal para medições de área no Brasil).
    """
    print("--> Reprojetando geometrias para sistema métrico plano (m²)...")
    grid_proj = gdf_grid.to_crs(crs_metrico)
    mun_proj = gdf_municipios.to_crs(crs_metrico)
    
    # Área total real de cada município do IBGE em m²
    print("--> Calculando áreas totais dos municípios...")
    mun_proj['area_municipio_m2'] = mun_proj.geometry.area
    
    print("--> Executando intersecção geométrica (Overlay Espacial)... isso pode levar de 1 a 3 minutos.")
    start_overlay = time.time()
    
    # Intersecção entre a malha e os municípios
    intersection = gpd.overlay(grid_proj, mun_proj, how='intersection')
    
    end_overlay = time.time()
    print(f"    Intersecção concluída em {end_overlay - start_overlay:.2f} segundos.")
    
    # Área do pedaço (fração) que sobrepõe
    intersection['area_interseccao_m2'] = intersection.geometry.area
    
    print("--> Normalizando os fatores de peso por área...")
    # FATOR DE PESO: Quanto dessa célula representa a área TOTAL do município
    # A soma de 'fator_peso_municipio' para um mesmo município será exatamente 1.0 (100%)
    intersection['fator_peso_municipio'] = (
        intersection['area_interseccao_m2'] / intersection['area_municipio_m2']
    )
    
    # Arredondar para evitar dízimas no Spark
    intersection['fator_peso_municipio'] = intersection['fator_peso_municipio'].round(6)
    
    # Filtrar pequenas ruínas de borda insignificantes (< 0.01% da área do município)
    intersection = intersection[intersection['fator_peso_municipio'] > 0.0001].copy()
    
    # Seleção final de colunas ajustadas para o Modelo Dimensional
    colunas_finais = [
        'id_geo',
        'latitude_centro',
        'longitude_centro',
        'code_muni',           # Código IBGE (ex: 3550308)
        'name_muni',           # Nome do Município (ex: São Paulo)
        'abbrev_state',        # UF (ex: SP)
        'area_municipio_m2',
        'area_interseccao_m2',
        'fator_peso_municipio'
    ]
    
    df_result = pd.DataFrame(intersection[colunas_finais])
    return df_result



In [5]:

tempo_inicio = time.time()
print("=== INICIANDO CONSTRUÇÃO DA LOOKUP TABLE ESPACIAL (IBGE x COPERNICUS) ===")


# 1. Baixar mapa dos municípios
gdf_ibge = geobr.read_municipality(code_muni="all", year=2022)


# 2. Obter limites e ALINHAR nos múltiplos exatos da Copernicus (0.25)
bounds = gdf_ibge.total_bounds # [min_lon, min_lat, max_lon, max_lat]

# Força o início e o fim a encaixarem exatamente nos quartos de grau (0.00, 0.25, 0.50, 0.75)
step = 0.25
min_lon = math.floor(bounds[0] * 4) / 4
min_lat = math.floor(bounds[1] * 4) / 4
max_lon = math.ceil(bounds[2] * 4) / 4
max_lat = math.ceil(bounds[3] * 4) / 4

print(f"Limites ajustados da grade: Lon [{min_lon} a {max_lon}], Lat [{min_lat} a {max_lat}]")


# 3. Construir a grade estática do Copernicus
print("\n[Passo 2/4] Construindo malha do Copernicus...")
gdf_copernicus = criar_grade_copernicus(
    min_lon=bounds[0] - 0.25,
    min_lat=bounds[1] - 0.25,
    max_lon=bounds[2] + 0.25,
    max_lat=bounds[3] + 0.25,
    step=0.25  # Alterar para 0.1 se seus dados Copernicus forem na resolução de 9 km
)

# 4. Processar a sobreposição de áreas (Overlay)
print("\n[Passo 3/4] Processando Overlay Espacial e Ponderação de Áreas...")
df_lookup = calcular_pesos_sobreposicao(gdf_copernicus, gdf_ibge)

# 5. Salvar arquivo Parquet
print("\n[Passo 4/4] Exportando tabela final...")
nome_arquivo = "d_lookup_grid_municipio_brasil.parquet"
df_lookup.to_parquet(nome_arquivo, index=False)

tempo_total = time.time() - tempo_inicio
print(f"\nCONCLUÍDO COM SUCESSO EM {tempo_total/60:.2f} MINUTOS!")
print(f"--> Tabela gerada: '{nome_arquivo}'")
print(f"--> Total de associações (Célula x Município): {len(df_lookup):,} linhas")

# Exibir amostra dos dados
print("\nAmostra dos dados gerados:")
print(df_lookup.head(10))

=== INICIANDO CONSTRUÇÃO DA LOOKUP TABLE ESPACIAL (IBGE x COPERNICUS) ===
Limites ajustados da grade: Lon [-74.0 a -28.75], Lat [-34.0 a 5.5]

[Passo 2/4] Construindo malha do Copernicus...
--> Gerando grade alinhada do Copernicus (Resolução: 0.25°)...
    Total de células geradas na grade: 29,256

[Passo 3/4] Processando Overlay Espacial e Ponderação de Áreas...
--> Reprojetando geometrias para sistema métrico plano (m²)...
--> Calculando áreas totais dos municípios...
--> Executando intersecção geométrica (Overlay Espacial)... isso pode levar de 1 a 3 minutos.
    Intersecção concluída em 5.88 segundos.
--> Normalizando os fatores de peso por área...

[Passo 4/4] Exportando tabela final...

CONCLUÍDO COM SUCESSO EM 0.16 MINUTOS!
--> Tabela gerada: 'd_lookup_grid_municipio_brasil.parquet'
--> Total de associações (Célula x Município): 34,326 linhas

Amostra dos dados gerados:
               id_geo  latitude_centro  longitude_centro  code_muni  \
0  GRID_-33.75_-53.50           -33.75 

In [6]:
df_grid_municipio = spark.read.parquet(r"C:\Marco Conti\Projetos\mais_einstein\Municipio\d_lookup_grid_municipio_brasil.parquet")
df_grid_municipio.printSchema()
df_grid_municipio.show(10, False)

root
 |-- id_geo: string (nullable = true)
 |-- latitude_centro: double (nullable = true)
 |-- longitude_centro: double (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- name_muni: string (nullable = true)
 |-- abbrev_state: string (nullable = true)
 |-- area_municipio_m2: double (nullable = true)
 |-- area_interseccao_m2: double (nullable = true)
 |-- fator_peso_municipio: double (nullable = true)

+------------------+---------------+----------------+---------+-----------------------+------------+-------------------+--------------------+--------------------+
|id_geo            |latitude_centro|longitude_centro|code_muni|name_muni              |abbrev_state|area_municipio_m2  |area_interseccao_m2 |fator_peso_municipio|
+------------------+---------------+----------------+---------+-----------------------+------------+-------------------+--------------------+--------------------+
|GRID_-33.75_-53.50|-33.75         |-53.5           |4305439.0|Chuí                   |RS     

In [ ]:
# df_grid_municipio.select("code_muni").dropDuplicates().count() # 34326 Total / 5570 Municípios
df_grid_municipio.filter("code_muni = 2605459").show()

In [7]:
df_temperatura = spark.read.csv(r"C:\Marco Conti\Projetos\Dados\ERA5-temperaturas\arquivos_CSV\ERA5_t2m_2025.csv", sep=",", header=True)
# df_temperatura = spark.read.csv(r"C:\Marco Conti\Projetos\Dados\ERA5-temperaturas\arquivos_CSV\ERA5_t2m_2026.csv", sep=",", header=True)
df_temperatura = \
    df_temperatura.withColumns({"latitude": F.col("latitude").cast("double")
                               ,"longitude": F.col("longitude").cast("double")})

In [ ]:
df_temperatura.printSchema()


In [ ]:
(df_temperatura
    .filter("""((latitude = -08.25 or latitude = -08.0) ) """)
    .select("latitude", "longitude")
    .dropDuplicates()
    .show(10, False)
)

# (longitude = -37 or longitude = 37.25)

In [ ]:
df_grid_municipio.select("code_muni").dropDuplicates().count() # 

In [8]:
# Faz a junção dos municípios e temperatura aplicando o peso ponderado para o município

df_grid_municipio.createOrReplaceTempView("d_lookup_grid_municipio")
df_temperatura.createOrReplaceTempView("temp_temperatura")


query_temp_SP = \
    """ SELECT l.code_muni,
            l.name_muni,
            l.abbrev_state AS uf,
            f.data_medicao,
            ROUND(SUM(f.valor * l.fator_peso_municipio), 2) AS temp_media_municipio
          FROM d_lookup_grid_municipio l 
          left JOIN temp_temperatura f
            -- Garante o encaixe perfeito das coordenadas numéricas
            ON ROUND(f.latitude,  2) = ROUND(l.latitude_centro, 2)
           AND ROUND(f.longitude, 2) = ROUND(l.longitude_centro,2)
         where 1=1
           and f.data_medicao = '2025-07-01'  
           --and name_muni = 'São Paulo'
         GROUP BY 
            l.code_muni, 
            l.name_muni, 
            l.abbrev_state, 
            f.data_medicao
    """


df_temp_SP = spark.sql(query_temp_SP)
df_temp_SP.printSchema()
df_temp_SP.show(10,False)


root
 |-- code_muni: double (nullable = true)
 |-- name_muni: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- data_medicao: string (nullable = true)
 |-- temp_media_municipio: double (nullable = true)

+---------+----------------------------+---+------------+--------------------+
|code_muni|name_muni                   |uf |data_medicao|temp_media_municipio|
+---------+----------------------------+---+------------+--------------------+
|2103703.0|Cururupu                    |MA |2025-07-01  |25.59               |
|2111201.0|São José de Ribamar         |MA |2025-07-01  |26.21               |
|1505486.0|Pacajá                      |PA |2025-07-01  |23.54               |
|2111409.0|São Luís Gonzaga do Maranhão|MA |2025-07-01  |25.09               |
|2311264.0|Quiterianópolis             |CE |2025-07-01  |22.66               |
|1502152.0|Canaã dos Carajás           |PA |2025-07-01  |22.94               |
|2407302.0|Marcelino Vieira            |RN |2025-07-01  |25.1          

In [12]:
df_temp_SP.select('code_muni').dropDuplicates().count()

5570

In [9]:
# 34326 / 1809305
# df_temp_SP.count()
# df_temp_SP.filter("temp_media_municipio is not null").count()
df_temp_SP.createOrReplaceTempView("temp_df_temp_SP")


q_check_left_join = \
    """Select g.code_muni, t.code_muni
         from d_lookup_grid_municipio g
         left join temp_df_temp_SP t 
           on g.code_muni = t.code_muni 
        where g.code_muni is not null 
          and t.code_muni is null
    """

spark.sql(q_check_left_join).show()

+---------+---------+
|code_muni|code_muni|
+---------+---------+
+---------+---------+



In [11]:
query_check_grid = \
    """
    SELECT 
        code_muni, 
        name_muni, 
        SUM(fator_peso_municipio) AS soma_pesos,
        COUNT(id_geo) AS qtd_pixels_intersectados
    FROM d_lookup_grid_municipio
    GROUP BY code_muni, name_muni
    HAVING SUM(fator_peso_municipio) < 0.98 OR SUM(fator_peso_municipio) > 1.02
    ORDER BY soma_pesos ASC;"""

query_check_grid2 = \
    """
    SELECT 
        l.name_muni,
        f.data_medicao,
        MIN(f.valor) AS temp_min_dos_pixels,
        ROUND(SUM(f.valor * l.fator_peso_municipio), 2) AS temp_media_calculada,
        MAX(f.valor) AS temp_max_dos_pixels
    FROM temp_temperatura f
    INNER JOIN d_lookup_grid_municipio l 
        ON ROUND(f.latitude, 2) = l.latitude_centro 
    AND ROUND(f.longitude, 2) = l.longitude_centro
    WHERE l.name_muni IN ('São Paulo', 'Altamira', 'Manaus') -- Teste cidades grandes
    GROUP BY l.name_muni, f.data_medicao;
    """

spark.sql(query_check_grid2).show()

+---------+------------+-------------------+--------------------+-------------------+
|name_muni|data_medicao|temp_min_dos_pixels|temp_media_calculada|temp_max_dos_pixels|
+---------+------------+-------------------+--------------------+-------------------+
| Altamira|  2025-01-01| 22.634667968750023|               23.71| 25.673730468750023|
| Altamira|  2025-01-02| 22.461083984375023|               23.95| 25.636865234375023|
| Altamira|  2025-01-03| 22.469628906250023|               23.86| 25.532128906250023|
| Altamira|  2025-01-04| 22.881982421875023|               24.67| 25.842919921875023|
| Altamira|  2025-01-05| 22.231835937500023|               23.34| 24.345117187500023|
| Altamira|  2025-01-06| 22.339501953125023|               23.49| 24.640283203125023|
| Altamira|  2025-01-07| 22.086816406250023|               23.69| 24.877832031250023|
| Altamira|  2025-01-08| 21.840478515625023|               23.41| 24.731103515625023|
| Altamira|  2025-01-09| 21.599267578125023|          